# Alphabet Recognition - PyTorch WITH DATA AUGMENTATION
Train with augmentation to improve accuracy from 56% to 70-80%

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.transforms import RandomRotation, RandomAffine, GaussianBlur
import numpy as np
import matplotlib.pyplot as plt
from bidict import bidict
from sklearn.utils import shuffle
import os
from PIL import Image

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Upload data files

In [ ]:
from google.colab import files
os.makedirs('data', exist_ok=True)

print("Upload images.npy and labels.npy")
uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith('.npy'):
        os.rename(filename, f'data/{filename}')
        print(f"✓ {filename}")

## Configuration & Load Data

In [ ]:
BATCH_SIZE = 32  # Larger batch size
EPOCHS = 50  # More epochs
LEARNING_RATE = 0.001  # Higher learning rate for better convergence
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ENCODER = bidict({
    'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5,
    'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11,
    'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17,
    'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23,
    'Y': 24, 'Z': 25
})

NUM_CLASSES = 26

# Load data
labels_raw = np.load('data/labels.npy', allow_pickle=True)
labels = np.array([ENCODER[x] for x in labels_raw])
imgs = np.load('data/images.npy').astype('float32') / 255.0

if len(imgs.shape) == 3:
    imgs = np.expand_dims(imgs, -1)

print(f"Dataset size: {len(imgs)}")
print(f"Image shape: {imgs.shape}")
print(f"Label range: {labels.min()}-{labels.max()}")

## Custom Dataset with Data Augmentation

In [ ]:
class AugmentedDataset(Dataset):
    """Dataset with data augmentation"""
    def __init__(self, imgs, labels, augment=False):
        self.imgs = torch.from_numpy(imgs).permute(0, 3, 1, 2).float()
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment
        
        # Augmentation transforms
        self.augmentation = transforms.Compose([
            transforms.RandomRotation(15),  # Rotate ±15 degrees
            transforms.RandomAffine(0, translate=(0.1, 0.1)),  # Shift 10%
            transforms.RandomAffine(0, scale=(0.9, 1.1)),  # Scale 90-110%
        ])
    
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        img = self.imgs[idx]
        label = self.labels[idx]
        
        if self.augment:
            img = self.augmentation(img)
        
        return img, label

# Split data
labels, imgs = shuffle(labels, imgs, random_state=42)
split = int(len(labels) * 0.75)

imgs_train, imgs_test = imgs[:split], imgs[split:]
labels_train, labels_test = labels[:split], labels[split:]

# Create datasets with augmentation for training
train_dataset = AugmentedDataset(imgs_train, labels_train, augment=True)
test_dataset = AugmentedDataset(imgs_test, labels_test, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## Improved Model Architecture

In [ ]:
class ImprovedAlphabetCNN(nn.Module):
    def __init__(self, num_classes=26):
        super(ImprovedAlphabetCNN, self).__init__()
        
        # Block 1: 1 -> 64 channels
        self.conv1a = nn.Conv2d(1, 64, 3, padding=1)
        self.conv1b = nn.Conv2d(64, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2)
        self.drop1 = nn.Dropout(0.25)
        
        # Block 2: 64 -> 128 channels
        self.conv2a = nn.Conv2d(64, 128, 3, padding=1)
        self.conv2b = nn.Conv2d(128, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2)
        self.drop2 = nn.Dropout(0.25)
        
        # Block 3: 128 -> 256 channels
        self.conv3a = nn.Conv2d(128, 256, 3, padding=1)
        self.conv3b = nn.Conv2d(256, 256, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d(2)
        self.drop3 = nn.Dropout(0.25)
        
        # Global average pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # Fully connected
        self.fc1 = nn.Linear(256, 128)
        self.bn_fc = nn.BatchNorm1d(128)
        self.drop_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # Block 1
        x = self.relu(self.conv1a(x))
        x = self.relu(self.bn1(self.conv1b(x)))
        x = self.pool1(x)
        x = self.drop1(x)
        
        # Block 2
        x = self.relu(self.conv2a(x))
        x = self.relu(self.bn2(self.conv2b(x)))
        x = self.pool2(x)
        x = self.drop2(x)
        
        # Block 3
        x = self.relu(self.conv3a(x))
        x = self.relu(self.bn3(self.conv3b(x)))
        x = self.pool3(x)
        x = self.drop3(x)
        
        # Global pooling + FC
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.bn_fc(self.fc1(x)))
        x = self.drop_fc(x)
        x = self.fc2(x)
        
        return x

model = ImprovedAlphabetCNN(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

## Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("✓ Loss, optimizer, and scheduler configured")

## Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, pred = torch.max(outputs, 1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(loader), correct / total

# Training
print(f"Starting training for {EPOCHS} epochs...\n")

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0
patience, patience_count = 5, 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = validate(model, test_loader, criterion, DEVICE)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    scheduler.step()
    
    if (epoch + 1) % 5 == 0 or epoch < 3:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {train_loss:.4f}/{val_loss:.4f} | Acc: {train_acc:.4f}/{val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_count = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            model.load_state_dict(torch.load('best_model.pth'))
            break

print(f"\n✓ Training complete!")
print(f"Best validation accuracy: {best_val_acc*100:.2f}%")

## Evaluate & Plot

In [ ]:
test_loss, test_acc = validate(model, test_loader, criterion, DEVICE)
print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_acc'], label='Train', linewidth=2)
ax1.plot(history['val_acc'], label='Val', linewidth=2)
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_title('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_loss'], label='Train', linewidth=2)
ax2.plot(history['val_loss'], label='Val', linewidth=2)
ax2.set_ylabel('Loss')
ax2.set_xlabel('Epoch')
ax2.set_title('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Save Model

In [ ]:
os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), 'models/letter.pth')
print("✓ Model saved")

import json
encoder_dict = {str(k): int(v) for k, v in ENCODER.items()}
with open('models/encoder.json', 'w') as f:
    json.dump(encoder_dict, f, indent=2)
print("✓ Encoder saved")

files.download('models/letter.pth')
files.download('models/encoder.json')
print("✓ Files downloaded")